## Fase 4.8.a — Estructurar el dato: generar retenciones_chaco.csv

**Objetivo:** convertir la cronología de derechos de exportación
(`docs/cronologia_retenciones.md` — 6 decretos, ya verificados contra
InfoLeg/BORA) en una tabla (`data/raw/retenciones_chaco.csv`), una fila
por ventana temporal y producto, para poder cruzarla con pandas contra la
serie mensual de Chaco en los próximos pasos (4.8.b en adelante).

**Script:** `src/fase4_generar_retenciones_csv.py`

---

### ⚠️ Bug encontrado y corregido durante este paso

Al revisar el script contra el PDF de la cronología, se encontraron **dos
solapamientos de fechas en el producto trigo** (no en soja ni maíz/sorgo,
que ya estaban bien encadenados):

1. La fila de la prórroga (decreto 439/2025) tenía `fecha_fin: 2026-03-31`,
   pero el decreto 877/2025 la reemplazó antes, el 2025-12-12 — quedaban casi
   4 meses con dos alícuotas de trigo vigentes "al mismo tiempo" (9,5% y
   7,5%). Se corrigió acortando esa fecha_fin.
2. Al corregir lo anterior, esa misma fila de 439/2025 seguía "tapando" los
   3 días de la ventana especial del decreto 682/2025 (0%, 23-25 de
   septiembre). Se corrigió partiendo la fila en dos, con el mismo criterio
   que ya se usaba en soja y maíz/sorgo para el decreto 526/2025 (dejar un
   hueco exacto para la ventana de 3 días).

Se agregó además una validación nueva (`_validar_sin_solapamientos`) que
la validación original (`duplicated` por `producto` + `fecha_inicio`) no
podía detectar, porque solo mira fechas de inicio idénticas — no rangos
que se superponen con fechas de inicio distintas. Esta validación nueva
fue la que efectivamente atajó el segundo bug antes de generar el CSV.

In [3]:
%run ../src/fase4_generar_retenciones_csv.py

CSV generado y validado: C:\Users\nairz\Documents\proyectoCampañasAgro\exportaciones-chaco\data\raw\retenciones_chaco.csv
Filas: 20 | Decretos distintos: 6
Confianza: {'media': 16, 'alta': 3, 'baja': 1}


In [4]:
import pandas as pd
df = pd.read_csv('../data/raw/retenciones_chaco.csv', sep=';')
df[df['producto'] == 'trigo'][['fecha_inicio', 'fecha_fin', 'alicuota_pct', 'decreto']]

,fecha_inicio,fecha_fin,alicuota_pct,decreto
2,2025-01-27,2025-06-30,9.5,38/2025
4,2025-07-01,2025-09-22,9.5,439/2025
5,2025-09-26,2025-12-11,9.5,439/2025
12,2025-09-23,2025-09-25,0.0,682/2025
16,2025-12-12,2026-06-03,7.5,877/2025
18,2026-06-04,NaN,5.5,423/2026


### 📌 Conclusión parcial — Fase 4.8.a

`retenciones_chaco.csv` generado y validado: 20 filas, 6 decretos, sin
solapamientos ni duplicados. Distribución de confianza: 16 "media", 3
"alta" (las tres ventanas de 682/2025, con 0% confirmado en texto
oficial), 1 "baja" (maíz/sorgo en 38/2025, alícuota sin confirmar).

El bug de trigo encontrado en este paso es un buen recordatorio de por
qué el plan de trabajo exige "validar antes de narrar" (4.8.b) en vez de
saltar directo al cruce completo — si esto no se hubiera revisado ahora,
el gráfico de 4.8.c habría mostrado trigo con una alícuota incorrecta
durante casi 4 meses.

Listo para 4.8.b: verificar si hay señal en la serie mensual de Chaco
alrededor de los cambios de retenciones más marcados (ej. sep-2025,
dic-2025).